# 04 — Modelos: Nicolas Rodriguez
Experimentos con **Random Forest** (árbol) y **MLPClassifier** (red neuronal densa) usando la mejor codificación encontrada en el notebook 02.

In [1]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
import mlflow.sklearn
from dotenv import load_dotenv
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

warnings.filterwarnings('ignore')
load_dotenv()

RANDOM_SEED = int(os.getenv('RANDOM_SEED', 42))
MLFLOW_TRACKING_URI    = os.getenv('MLFLOW_TRACKING_URI', 'http://ec2-52-203-38-164.compute-1.amazonaws.com:5000')
MLFLOW_EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'sentiment140')
MEMBER = 'nicolas'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
np.random.seed(RANDOM_SEED)

DATA_ENCODED = Path('../data/encoded')
MODELS_DIR   = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete — member:', MEMBER)

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Setup complete — member: nicolas


## Carga de datos

Usamos TF-IDF bigrama (`tfidf_bi`) como encoding — cambiar a `tfidf_uni` si ese fue el mejor en notebook 02.

In [2]:
# tfidf_uni: ~50k features vs ~300k de tfidf_bi → mucho más rápido para el MLP
ENCODING = 'tfidf_uni'

X_train_full = sp.load_npz(DATA_ENCODED / f'X_train_{ENCODING}.npz')
X_test       = sp.load_npz(DATA_ENCODED / f'X_test_{ENCODING}.npz')
y_train_full = np.load(DATA_ENCODED / 'y_train.npy')
y_test       = np.load(DATA_ENCODED / 'y_test.npy')

# 10 % del train se reserva como validación (estratificado)
VAL_SIZE = 0.10
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=VAL_SIZE, random_state=RANDOM_SEED, stratify=y_train_full
)

print(f'Encoding : {ENCODING}')
print(f'Train    : {X_train.shape}')
print(f'Val      : {X_val.shape}')
print(f'Test     : {X_test.shape}')

Encoding : tfidf_uni
Train    : (1224000, 50000)
Val      : (136000, 50000)
Test     : (240000, 50000)


In [3]:
def _metrics(y_true, y_pred, prefix: str) -> dict:
    """Compute accuracy/precision/recall/f1 and prefix all keys."""
    return {
        f'{prefix}_accuracy':  accuracy_score(y_true, y_pred),
        f'{prefix}_precision': precision_score(y_true, y_pred, average='macro'),
        f'{prefix}_recall':    recall_score(y_true, y_pred, average='macro'),
        f'{prefix}_f1_macro':  f1_score(y_true, y_pred, average='macro'),
    }


def log_run(run_name: str, clf, params: dict, encoding: str,
            X_tr=None, X_v=None, X_te=None) -> dict:
    """Train, evaluate (train / val / test) and log a classifier to MLflow.

    X_tr / X_v / X_te permiten pasar datos distintos a los globales (e.g. tras
    TruncatedSVD). Si son None se usan las matrices globales X_train/X_val/X_test.

    Returns a dict with all metrics, the fitted clf, and test predictions.
    """
    _X_tr = X_tr if X_tr is not None else X_train
    _X_v  = X_v  if X_v  is not None else X_val
    _X_te = X_te if X_te is not None else X_test

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag('member', MEMBER)
        mlflow.set_tag('stage', 'models')
        mlflow.set_tag('model', type(clf).__name__)
        mlflow.set_tag('encoding', encoding)

        mlflow.log_params({'model': type(clf).__name__, 'encoding': encoding,
                           'random_seed': RANDOM_SEED, **params})

        clf.fit(_X_tr, y_train)

        train_preds = clf.predict(_X_tr)
        val_preds   = clf.predict(_X_v)
        test_preds  = clf.predict(_X_te)

        train_metrics = _metrics(y_train, train_preds, 'train')
        val_metrics   = _metrics(y_val,   val_preds,   'val')
        test_metrics  = _metrics(y_test,  test_preds,  'test')

        # MLflow only stores train + test; val is used locally for model selection
        mlflow.log_metrics({**train_metrics, **test_metrics})
        mlflow.sklearn.log_model(clf, 'model')

        # Log model card artifact
        model_card = {
            'model_name': type(clf).__name__,
            'encoding': encoding,
            'parameters': params,
            'member': MEMBER,
            'stage': 'models',
            'task': 'Binary Sentiment Classification',
            'dataset': 'Sentiment140',
            'metrics': {
                'train': {k.replace('train_', ''): round(v, 4) for k, v in train_metrics.items()},
                'val':   {k.replace('val_',   ''): round(v, 4) for k, v in val_metrics.items()},
                'test':  {k.replace('test_',  ''): round(v, 4) for k, v in test_metrics.items()},
            },
            'created_at': pd.Timestamp.now().isoformat(),
        }
        mlflow.log_dict(model_card, 'model_card.json')

        run_id = run.info.run_id

    result = {**train_metrics, **val_metrics, **test_metrics}
    result['run_id']      = run_id
    result['model']       = type(clf).__name__
    result['run_name']    = run_name
    result['params']      = params
    result['test_preds']  = test_preds
    result['clf']         = clf
    return result

## Modelo 1 — Random Forest (grid de hiperparámetros)

Se prueba un grid de `max_depth` (profundidad del árbol) con valores 10, 25, 50 y 100. Para cada configuración se registran métricas de **train**, **validación** y **test** en MLflow y se guarda el modelo en disco.

In [ ]:
RF_MAX_DEPTHS = [10, 25, 50, 75]

rf_all_results = []

for depth in RF_MAX_DEPTHS:
    params = {
        'n_estimators': 200,
        'max_depth': depth,
        'min_samples_split': 5,
        'n_jobs': -1,
        'random_state': RANDOM_SEED,
    }
    run_name = f'rf_{ENCODING}_d{depth}'
    clf = RandomForestClassifier(**params)
    res = log_run(run_name, clf, params, ENCODING)
    rf_all_results.append(res)

    print(f'max_depth={depth:4}  '
          f'train_f1={res["train_f1_macro"]:.4f}  '
          f'val_f1={res["val_f1_macro"]:.4f}  '
          f'test_f1={res["test_f1_macro"]:.4f}  '
          f'run_id={res["run_id"]}')

# Mejor RF según F1 en validación
rf_best = max(rf_all_results, key=lambda x: x['val_f1_macro'])
print(f'\nMejor RF → max_depth={rf_best["params"]["max_depth"]}  '
      f'val_f1={rf_best["val_f1_macro"]:.4f}')

## Modelo 2 — MLP (Red Neuronal Densa, grid de hiperparámetros)

Se barre el parámetro de regularización `alpha` con distintas arquitecturas de capas ocultas. Métricas de train / val / test para cada combinación.

In [5]:
# ── Solución C: TruncatedSVD ────────────────────────────────────────────────
# MLPClassifier no acepta matrices sparse nativamente: las convierte a densas
# internamente, lo que con 50k+ features explota en tiempo y memoria.
# TruncatedSVD (LSA) comprime la representación a un espacio denso pequeño
# antes de entrenar la red, reduciendo el coste de O(N·F) a O(N·K) con K<<F.
SVD_COMPONENTS = 300

print(f'Reduciendo dimensión: {X_train.shape[1]:,} → {SVD_COMPONENTS} componentes...')
svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_SEED)

X_train_mlp = svd.fit_transform(X_train)
X_val_mlp   = svd.transform(X_val)
X_test_mlp  = svd.transform(X_test)

var_explained = svd.explained_variance_ratio_.sum()
print(f'SVD listo. Shape: {X_train_mlp.shape}  |  Varianza explicada: {var_explained:.2%}')
joblib.dump(svd, MODELS_DIR / f'svd_{ENCODING}_{SVD_COMPONENTS}.pkl')
print(f'SVD guardado en {MODELS_DIR / f"svd_{ENCODING}_{SVD_COMPONENTS}.pkl"}')

Reduciendo dimensión: 50,000 → 300 componentes...
SVD listo. Shape: (1224000, 300)  |  Varianza explicada: 37.44%
SVD guardado en ../models/svd_tfidf_uni_300.pkl


In [6]:
# Grid: combinaciones de arquitectura y regularización
# El MLP recibe los datos ya reducidos por SVD (X_train_mlp, X_val_mlp, X_test_mlp)
MLP_GRID = [
    {'hidden_layer_sizes': (128,),       'alpha': 1e-4}
]

mlp_all_results = []

for grid_params in MLP_GRID:
    params = {
        **grid_params,
        'activation': 'relu',
        'solver': 'adam',
        'batch_size': 512,
        'max_iter': 50,
        'early_stopping': True,
        'validation_fraction': 0.1,
        'random_state': RANDOM_SEED,
    }
    layers_str = 'x'.join(str(u) for u in grid_params['hidden_layer_sizes'])
    alpha_str  = f"{grid_params['alpha']:.0e}".replace('-0', 'm')
    run_name   = f'mlp_{ENCODING}_svd{SVD_COMPONENTS}_{layers_str}_a{alpha_str}'
    clf = MLPClassifier(**params)
    res = log_run(run_name, clf, params, f'{ENCODING}+svd{SVD_COMPONENTS}',
                  X_tr=X_train_mlp, X_v=X_val_mlp, X_te=X_test_mlp)
    mlp_all_results.append(res)

    print(f'layers={layers_str:<12}  alpha={grid_params["alpha"]:.0e}  '
          f'train_f1={res["train_f1_macro"]:.4f}  '
          f'val_f1={res["val_f1_macro"]:.4f}  '
          f'test_f1={res["test_f1_macro"]:.4f}  '
          f'run_id={res["run_id"]}')

# Mejor MLP según F1 en validación
mlp_best = max(mlp_all_results, key=lambda x: x['val_f1_macro'])
best_layers = 'x'.join(str(u) for u in mlp_best['params']['hidden_layer_sizes'])
print(f'\nMejor MLP → layers={best_layers}  alpha={mlp_best["params"]["alpha"]:.0e}  '
      f'val_f1={mlp_best["val_f1_macro"]:.4f}')

2026/03/09 07:43:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 07:43:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run mlp_tfidf_uni_svd300_128_a1em4 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/5be5a57b721c421a8a50b13e9e8844d8
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
layers=128           alpha=1e-04  train_f1=0.7947  val_f1=0.7755  test_f1=0.7752  run_id=5be5a57b721c421a8a50b13e9e8844d8

Mejor MLP → layers=128  alpha=1e-04  val_f1=0.7755
